**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**01 | Source and Reference Eligibility**

![Source and Reference Eligibility](../../images/01_source_and_reference_eligibility.svg)

**Objective:** Inspect the original Camargo sensor inventory, potential same-trial IMU/force pairing and physical reference feasibility before any model fitting.

**Dataset:** Camargo et al. (2021), treadmill/level-ground walking where original-time IMU and independent force reference are usable.

**Notebook scope:** Implemented analysis code; original-data results will be added only after execution and source eligibility confirmation.


**00. Colab Repository Setup**


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/human-movement-intelligence


**01. Imports and Research Environment**

Initialise shared Colab utilities, the study-specific source adapter and source/output paths. Original source files are expected under the repository data directory or a caller-supplied mounted-data path; downloading large archives is not automatic.


In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("gait_analysis")
NOTEBOOK_NUMBER = 1

# ----------------------- 2. MATLAB Table Reader Setup ------------------------
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("matio") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "mat-io==1.0.2"],
        check=True,
    )


# ----------------- 3. Scientific Computing and Data Analysis -----------------
import os
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 5. Study Research Utilities ------------------------
from camargo_source import (
    inventory_files, list_archives, paired_trials,
    read_sensor_table, validate_timebase, inspect_columns,
    extract_pilot_trial_from_zip, locate_pilot_source_archive,
    download_mendeley_pilot_archive,
)
from gait_alignment import candidate_channels, preview_channels

# ---------------------- 6. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
source_root = Path(os.environ.get(
    "CAMARGO_DATA_ROOT", root / "data/raw/camargo"
))
output_dir = study_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print("Original source root:", source_root)
print("Scientific output folder:", output_dir)


Repository updated.
Original source root: /content/human-movement-intelligence/data/raw/camargo
Scientific output folder: /content/human-movement-intelligence/gait_analysis/output


**02. Prepare One Original Camargo Participant for Source Inspection**

The official Camargo dataset is not stored in GitHub. Version 1 of [Mendeley Part 1](https://data.mendeley.com/datasets/fcgm3chfff/1) contains a separate **AB06.zip** (approximately 1.12 GB). Version 2 combines nine subjects into a much larger archive.

**Default:** `AUTO_DOWNLOAD_PILOT = True` downloads only `AB06.zip` directly from Mendeley **when no extracted data or local ZIP is found**. The transfer uses the publisher's version-1 file metadata, validates the downloaded ZIP and extracts only one matched treadmill IMU/force trial. Network access and sufficient Colab disk space are required. The download is never copied to GitHub and may have to be repeated after the runtime is reset.

If you already have `AB06.zip`, place it in `/content/`, or set `MOUNT_GOOGLE_DRIVE = True` and `PILOT_ZIP_PATH` to its mounted Drive location. These existing copies take precedence; to disable remote transfer set `AUTO_DOWNLOAD_PILOT = False`. If the public Mendeley API is blocked, use this manual/Drive route instead.

This step only obtains real original sensor files. It does **not** independently validate the IMU-force timebase, side mapping or force-contact labels.


In [3]:
# ==================== 02. Prepare Original Camargo Pilot =====================

# ---------------- 1. Find Original Source or Participant ZIP -----------------
PILOT_SUBJECT = "AB06"
MOUNT_GOOGLE_DRIVE = False
PILOT_ZIP_PATH = None  # e.g. "/content/drive/MyDrive/Camargo/AB06.zip"
AUTO_DOWNLOAD_PILOT = True  # fetch ~1.12 GB from Mendeley v1 if absent

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

existing = (inventory_files(source_root) if source_root.is_dir()
            else pd.DataFrame())
available_pairs = paired_trials(existing)
usable = available_pairs.loc[available_pairs.candidate_pair] if not (
    available_pairs.empty
) else available_pairs
if not usable.empty:
    print("Reusing existing matched Camargo IMU/force files:", source_root)
else:
    pilot_archive = locate_pilot_source_archive(
        source_root, subject_id=PILOT_SUBJECT,
        explicit_archive=PILOT_ZIP_PATH,
    )
    if pilot_archive is None:
        if AUTO_DOWNLOAD_PILOT:
            pilot_archive = download_mendeley_pilot_archive(
                source_root, filename="AB06.zip",
                dataset_id="fcgm3chfff", version=1,
            )
        else:
            raise FileNotFoundError(
                "No original Camargo data found. Set AUTO_DOWNLOAD_PILOT "
                "to True to fetch only AB06.zip from Mendeley Version 1, "
                "or set PILOT_ZIP_PATH to an existing local/Drive archive."
            )

# --------------------- 2. Extract Only One Paired Trial ----------------------
    pilot = extract_pilot_trial_from_zip(
        pilot_archive, source_root, subject_id=PILOT_SUBJECT,
        condition="treadmill",
    )
    print("Extracted real source trial:", pilot)

# ------------------------- 3. Verify Original Files --------------------------
pilot_inventory = inventory_files(source_root)
print("Observed sensor files:", len(pilot_inventory))
print("Pilot participants:", sorted(pilot_inventory.subject_id.unique()))
if pilot_inventory.empty:
    raise FileNotFoundError("No original ABxx sensor files discovered.")


Reusing existing matched Camargo IMU/force files: /content/human-movement-intelligence/data/raw/camargo
Observed sensor files: 2
Pilot participants: ['AB06']


**03. Inspect Original Archive and Sensor Inventory**

Enumerate the extracted original files without forcing full-population download. The canonical Camargo release contains multiple sensors and locomotion modes; the inventory records what is present locally, not an assumed 22-person analysis cohort.


In [4]:
# ======================= 03. Original Source Inventory =======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
if not source_root.exists():
    raise FileNotFoundError(
        f"No Camargo source found at {source_root}. "
        "Download/extract one participant (ABxx) from the official Georgia Tech "
        "Camargo dataset, or set CAMARGO_DATA_ROOT to the mounted data folder."
    )
archives = list_archives(source_root)
inventory = inventory_files(source_root)
display(archives.head(10), inventory.head(15))
print("Observed subjects:", inventory.subject_id.nunique())
print("Observed sensor records:", inventory.sensor.value_counts().to_dict())

# ------------------------- 2. Save Scientific Tables -------------------------
inventory.to_csv(output_dir / "01_trial_inventory.csv", index=False)
if inventory.empty:
    raise ValueError(
        "No extracted ABxx/.../imu or fp source records were discovered. "
        "ZIP files alone are not decoded; inspect archive structure and "
        "extract one participant before proceeding."
    )


,archive_name,member_count,size_bytes
0,AB06.zip,1576,1202725820


,subject_id,session_id,condition,sensor,trial_id,relative_path,format,size_bytes
0,AB06,10_09_18,treadmill,fp,treadmill_01_01,AB06/10_09_18/treadmill/fp/treadmill_01_01.mat,.mat,11367266
1,AB06,10_09_18,treadmill,imu,treadmill_01_01,AB06/10_09_18/treadmill/imu/treadmill_01_01.mat,.mat,5323460


Observed subjects: 1
Observed sensor records: {'fp': 1, 'imu': 1}


**04. Identify Candidate Paired Trials**

Pair IMU and force measurements only when participant, session, condition and trial identifiers agree. These are candidate pairs, not yet independently validated synchronisation.


In [5]:
# ===================== 04. Identify Paired Sensor Trials =====================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
pairs = paired_trials(inventory)
display(pairs.head(15))
print("Candidate IMU/FP pairs:", int(pairs.candidate_pair.sum()))
pairs.to_csv(output_dir / "01_candidate_pairs.csv", index=False)
if not pairs.candidate_pair.any():
    raise ValueError(
        "No same-trial IMU/force pairs were detected. Inspect source paths "
        "and exact trial-name conventions before extending the adapter."
    )


,subject_id,session_id,condition,trial_id,imu_path,fp_path,candidate_pair,pair_issue
0,AB06,10_09_18,treadmill,treadmill_01_01,AB06/10_09_18/treadmill/imu/treadmill_01_01.mat,AB06/10_09_18/treadmill/fp/treadmill_01_01.mat,True,


Candidate IMU/FP pairs: 1


**05. Inspect One Original Trial and Channel Headers**

Decode one genuine candidate pair and inspect the observed columns. No IMU axis, vertical-force direction or foot-side mapping is assumed from filenames. MATLAB table layouts may require an explicit validated export before they can be read.


In [6]:
# ========================= 05. Inspect Trial Headers =========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
sample = pairs.loc[pairs.candidate_pair].iloc[0]
imu_file = source_root / sample.imu_path
fp_file = source_root / sample.fp_path
imu_table = read_sensor_table(imu_file)
fp_table = read_sensor_table(fp_file)
imu_time = validate_timebase(imu_table, "IMU")
fp_time = validate_timebase(fp_table, "Force")
column_audit = pd.concat([
    inspect_columns(imu_table, imu_file).assign(modality="imu"),
    inspect_columns(fp_table, fp_file).assign(modality="fp"),
], ignore_index=True)
display(column_audit)

# ------------------------- 2. Save Scientific Tables -------------------------
column_audit.to_csv(output_dir / "01_observed_columns.csv", index=False)
print("IMU likely gyro/segment field names:", candidate_channels(imu_table, "imu"))
print("Potential force field names:", candidate_channels(fp_table, "fp"))
print("Native IMU time step (source units):", np.median(np.diff(imu_time)))
print("Native Force time step (source units):", np.median(np.diff(fp_time)))


,relative_file,column,dtype,numeric_fraction,min_observed,max_observed,modality
0,/content/human-movement-intelligence/data/raw/...,Header,float64,1.0,11.990000,155.045000,imu
1,/content/human-movement-intelligence/data/raw/...,foot_Accel_X,float64,1.0,-7.956601,7.916010,imu
2,/content/human-movement-intelligence/data/raw/...,foot_Accel_Y,float64,1.0,-3.249134,5.717045,imu
3,/content/human-movement-intelligence/data/raw/...,foot_Accel_Z,float64,1.0,-4.841972,7.641023,imu
4,/content/human-movement-intelligence/data/raw/...,foot_Gyro_X,float64,1.0,-6.846229,7.901357,imu
5,/content/human-movement-intelligence/data/raw/...,foot_Gyro_Y,float64,1.0,-8.191790,11.191073,imu
6,/content/human-movement-intelligence/data/raw/...,foot_Gyro_Z,float64,1.0,-8.713423,4.861218,imu
7,/content/human-movement-intelligence/data/raw/...,shank_Accel_X,float64,1.0,-2.267094,0.188931,imu
8,/content/human-movement-intelligence/data/raw/...,shank_Accel_Y,float64,1.0,-0.510689,0.696286,imu
9,/content/human-movement-intelligence/data/raw/...,shank_Accel_Z,float64,1.0,-1.206570,1.010313,imu


IMU likely gyro/segment field names: ['foot_Accel_X', 'foot_Accel_Y', 'foot_Accel_Z', 'foot_Gyro_X', 'foot_Gyro_Y', 'foot_Gyro_Z', 'shank_Accel_X', 'shank_Accel_Y', 'shank_Accel_Z', 'shank_Gyro_X', 'shank_Gyro_Y', 'shank_Gyro_Z', 'thigh_Gyro_X', 'thigh_Gyro_Y', 'thigh_Gyro_Z', 'trunk_Gyro_X', 'trunk_Gyro_Y', 'trunk_Gyro_Z']
Potential force field names: []
Native IMU time step (source units): 0.0049999999999954525
Native Force time step (source units): 0.0010000000000047748


**06. Visualise Recorded IMU and Treadmill Source Channels**

The actual AB06 source fields include `foot_Gyro_Y`, `shank_Gyro_Y`, `Treadmill_R_vy` and `Treadmill_L_vy`. The previous force panel was empty because the candidate matcher did not recognise the released `Treadmill_*_v*` names, **not** because the force-platform signal was absent.

The preview now plots short recorded intervals from the IMU and treadmill tables, using their own stored `Header` coordinates. The `v_y` signals are displayed with their original names. Their vertical-axis convention, physical units, sign, foot-side correspondence and cross-device synchronisation have **not** been verified. No contact labels are inferred during source inspection.


In [ ]:
# ======================= 06. Visualise Source Signals ========================

# ------------------ 1. Choose Observed Source Channel Names ------------------
imu_channels = preview_channels(imu_table, "imu")
force_channels = preview_channels(fp_table, "fp")
if not imu_channels or not force_channels:
    raise ValueError("Insufficient numeric sensor columns for preview.")
print("IMU gyroscope channels:", imu_channels)
print("Treadmill force-platform components:", force_channels)
print("These plotted channels are not yet validated contact references.")

# --------------- 2. Plot Short Recorded Native-Time Intervals ----------------
fig, axes = plt.subplots(
    2, 1, figsize=(11, 6.4), sharex=False, layout="constrained"
)
panels = (
    (axes[0], imu_table, imu_time, imu_channels,
     "Observed IMU gyroscope", ("#236CDE", "#168C5B")),
    (axes[1], fp_table, fp_time, force_channels,
     "Observed treadmill force-platform components",
     ("#E87929", "#236CDE")),
)
for ax, table, time, columns, title, colours in panels:
    native_time = np.asarray(time, dtype=float)
    view = native_time <= native_time[0] + 5.0
    if np.count_nonzero(view) < 3:
        raise ValueError("Too few native-time samples for signal preview.")
    for column, colour in zip(columns[:2], colours):
        signal = pd.to_numeric(table.loc[view, column], errors="coerce")
        ax.plot(native_time[view], signal, color=colour,
                linewidth=1.0, label=column)
    ax.set_title(title, loc="left")
    ax.set_ylabel("Channel value (source units)")
    ax.set_xlabel("Stored Header (source time units)")
    ax.legend(loc="upper right", ncol=2, frameon=False)
    ax.grid(alpha=0.2)
fig.suptitle(
    f"{sample.subject_id} / {sample.trial_id}: recorded sensor signals"
)

# --------------------- 3. Export Observed Signal Preview ---------------------
save_figure(fig, output_dir / "01_sensor_source_preview.png")
plt.show()


**07. Summarise Reference Eligibility and Next Dependency**

Record the source-level status without assigning scientific eligibility based solely on matching filenames. Notebook 02 must confirm original-time alignment, force side, vertical-force sign, and a reliable channel map before producing event references.


In [8]:
# ===================== 07. Summarise Source Eligibility ======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
summary = pd.DataFrame([
    {"item": "source_participants_observed",
     "value": int(inventory.subject_id.nunique())},
    {"item": "candidate_imu_force_pairs",
     "value": int(pairs.candidate_pair.sum())},
    {"item": "validated_reference_pairs",
     "value": "pending original-time/channel/side inspection"},
    {"item": "force_contact_event_results",
     "value": "not generated in source inspection"},
])
display(summary)

# ------------------------- 2. Save Scientific Tables -------------------------
summary.to_csv(output_dir / "01_source_eligibility_summary.csv", index=False)
print("Next: review 01_observed_columns.csv and paired trial paths; "
      "then configure Notebook 02 using verified source units and side mapping.")


,item,value
0,source_participants_observed,1
1,candidate_imu_force_pairs,1
2,validated_reference_pairs,pending original-time/channel/side inspection
3,force_contact_event_results,not generated in source inspection


Next: review 01_observed_columns.csv and paired trial paths; then configure Notebook 02 using verified source units and side mapping.


**08. Research Summary and Output Download**

The recorded AB06 source inspection contains original channel names and a candidate paired IMU/force trial. Physical axis orientation, force sign and shared timebase have not yet been independently validated. Notebook 02 must review these before event labelling.

The shared helper packages the saved `01_` research tables and figures into one ZIP **outside the repository**. Retrieve it from Colab's file browser. Raw Camargo archives are not included, and no output is pushed to GitHub automatically.


In [ ]:
# ================= 08. Research Summary and Output Download ==================

# ------------------- 1. Review Current Scientific Outputs --------------------
print('Source inventory and signal preview saved to:', output_dir)

# ------------------- 2. Package Notebook-Specific Evidence -------------------
download_outputs_from_colab(
    folder="gait_analysis/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
